# Multifactor Models and Multivariate Regression

This notebook covers two related topics. The first reviews multifactor models: extensions of the basic CAPM that include additional sources of priced risk beyond the market factor. The second is multivariate regression, the linear-regression workhorse needed to estimate and test multifactor models, and a critical foundation for more advanced FinTech topics such as time series models, forecasting, and machine learning.

This notebook builds directly on `CAPM_BivariateRegression.ipynb`, picking up right where that notebook left off: with the CAPM's central prediction failing in the data.

*Acknowledgment*: This notebook was produced with the assistance of AI. I remain responsible for the content and any errors.


## Learning Objectives

By the end of this notebook, you should be able to:

1. Define a multifactor model and explain how it extends the CAPM by adding additional sources of priced risk (e.g., the `SMB` and `HML` factors in the Fama-French three-factor model).
2. State the key implication of a multifactor model for expected returns, and explain why testing it requires a multivariate (rather than bivariate) time-series regression.
3. Write down the multivariate linear regression model in both scalar/summation and matrix form, and derive the OLS estimator $\hat\beta = (X'X)^{-1}X'Y$.
4. Decompose total sum of squares into explained and residual components ($TSS = ESS + RSS$) and compute both uncentered and centered $R^2$.
5. State the large-sample statistical properties of the multivariate OLS estimator (unbiasedness, consistency, asymptotic normality) and the assumptions needed for them to hold.
6. Estimate multifactor (market model and three-factor) regressions in Python using `statsmodels`, and interpret the resulting alpha and factor-beta estimates.
7. Apply multivariate regression to test whether the CAPM and the Fama-French three-factor model can "price" a set of investment-sorted portfolios, and interpret evidence of model rejection (or non-rejection) via estimated alphas.
8. Explain why the three-factor model fails to explain momentum returns, and connect this to further extensions (Carhart's four-factor model, and the profitability, investment, and "quality minus junk" factors).


## Instructions to Run

**Data:** This notebook pulls four datasets live from Kenneth French's Data Library via `pandas_datareader`: the "F-F Research Data Factors" (market excess return `Mkt-RF`, the `SMB` and `HML` factors, and the risk-free rate `RF`), the "10 Industry Portfolios," the investment/asset-growth-sorted portfolios ("Portfolios_Formed_on_INV"), and the "F-F Momentum Factor." **An internet connection is required to run this notebook.** If a pull is slow or fails, French's site is occasionally unreachable or rate-limited -- wait a moment and re-run the cell. Each run also writes a timestamped CSV backup of the raw pulled data to `../data/ff_data_backup/` (a subfolder of this repo's `data/` directory, which is untracked by git by default -- see `.gitignore` and the README's "A note on data") so that a specific historical pull can be recovered later if French revises the underlying figures.

**Package dependencies:** `numpy`, `pandas`, `matplotlib`, `statsmodels`, and `pandas_datareader` -- all but the last are standard scientific-Python packages included in a typical Anaconda install. Otherwise: `pip install pandas numpy matplotlib statsmodels pandas_datareader`.

**Viewing the HTML export in Canvas:** If the equations below show up as raw text (things like `$\alpha$`) instead of typeset math, don't use Canvas's built-in file preview -- download the file and open it directly in your browser instead. Canvas's preview sandboxes uploaded HTML and blocks the script that typesets the math; opening the downloaded file locally has no such restriction.


## Motivating Example: Where the CAPM Left Off

Recall from `CAPM_BivariateRegression.ipynb` that testing the CAPM using the 25 Fama-French size- and book-market-sorted portfolios revealed a real empirical failure: 10 of the 25 estimated market-model alphas were statistically significant at the 5% level, and several were economically large -- on the order of several percentage points per year. If the CAPM were literally true, none of these alphas should differ from zero; market beta alone should explain all cross-sectional variation in expected returns.

This notebook asks whether adding more sources of priced risk beyond the market factor can close that gap. We will build up the relevant theory -- multifactor models, and the multivariate regression machinery needed to estimate them -- and then put it to work on three concrete examples later in the notebook: (1) a single industry portfolio (nondurables), (2) a set of portfolios sorted on investment/asset growth, a well-known "anomaly" the three-factor model was not originally designed to explain, and (3) the momentum factor, which turns out to be a case where the three-factor model fails badly. By the end, you will have seen both a case where a multifactor model works well and a case where it doesn't -- itself an important lesson about the limits of any specific asset pricing model.


## Multifactor models

In the classic CAPM there is a single factor: the excess return on the market portfolio. Remember that, under the model, it is (only) covariance with this factor return that generates "priced risk" (higher expected returns) for a given stock or portfolio. 

The CAPM is an elegant model with a simple, but plausible, economic foundation. However, as we realized in our empirical exercise last time, the key predictions of the CAPM (zero alphas for any stock or portfolio) appear to be violated in the data.

Multifactor models extend the CAPM by adding more sources of priced risk to the model. 

One of the most famous multifactor models was proposed by Eugene Fama and Ken French in 1993. This "Fama-French three factor model" consists of three distinct sources of priced risk:

1. The market excess return (as in the CAPM)
2. The return on a "hedge portfolio" that is long small cap stocks and short large cap stocks. This factor is called "SMB" (for 'small minus big').
3. The return on a different hedge portfolio that is long value stocks and short growth stocks (after size balancing). This factor is called "HML" (for 'High (B/M) Minus Low (B/M)").


We can capture exposure to the different sources of risk in a multifactor model the same way we do for the CAPM: via the coefficients in a (time series) regressions of security or portfolio returns on the factor returns. The key difference is that, instead of a "bivariate" linear regression, we are specifying a "multivariate" linear regression. Therefore, instead of a single beta, we have multiple betas: one for each factor. 

For the Fama-French three factor model we can specify the following **multivariate regression model** for some stock or portfolio return:

$R_{t} = \alpha + \beta_{MKT} MKT_{t} + \beta_{SMB} SMB_{t} + \beta_{HML} HML_{t} + \epsilon_{t}$

Here MKT is the excess return on the market, and SMB and HML denote the returns in period $t$ for the size and value factors in the model. There are three different betas, one for each of the three factors in the model.  

There are other popular factors that are often included in multifactor models. One is the "momentum factor," which is the return on a portfolio that buys "winners" (top quartile or decile of stock performers over the most recent 12 months excluding the latest month) and sells "losers" (bottom quartile or decile of performers over the same period). Other proposed factors include:

1. An "investment factor": Buys firms with relatively *low* investment (asset growth) and shorts firms with relatively high asset growth
2. A "profitability factor": Buys relatively profitable firms and shorts relatively unprofitable firms
3. A "quality minus junk" factor: Yep, you guessed it, buys "quality" firms and sells "junk" where the real issue is how these qualities are measured, which is somewhat involved but described in a research [paper] (https://www.aqr.com/Insights/Research/Working-Paper/Quality-Minus-Junk).

More complicated models can be specified by extending the multivariate regression model equation above to incorporate whichever additional factors we think might be important.




## Multifactor models: Key implications

Let us assume for the moment that, instead of the CAPM, a multifactor model holds exactly. In order to be concrete, we can assume that the FF 3 factor model discussed previously, but it should be understood that the comments here apply to any multifactor model that holds.

Remember that, under the CAPM, the key equation describing expected returns is:
$E(R_{i}) = R_{f} + \beta_{i} E(MKT)$

*Note*: remember that MKT is the *excess return* on the market portfolio, i.e., $MKT = R_{MKT} - R_{f}$ so that $E(MKT)$ is the *market risk premium*.

*BUT* if instead the FF 3-factor model holds, then the above equation is altered in a natural way. The new equation describing expected returns under this multifactor model is:

$E(R_{i}) = R_{f} + \beta_{i,MKT} E(MKT) + \beta_{i,SMB} E(SMB) + \beta_{i,HML}E(HML)$

Important points:

1. There are now 3 relevant betas that generate variation in expected returns across assets. There is the market beta (same as in CAPM), but also the asset's beta with respect to the SMB factor and the asset's beta with respect to the HML factor.
2. These betas should be understood as the (population) slope coefficients in a regression of excess returns for asset $i$ onto returns for the MKT, SMB, and HML factors *jointly*, i.e., this is a *multivariate regression*. This is the reason that we will review multivariate regression below.
3. In determining expected returns, each beta is multiplied by the *premium* associated with the corresponding factor. The market premium is the compensation investors require for bearing market risk (as in CAPM). The SMB premium is the compensation for a risk distinct from market risk that is captured by the performance of the SMB portfolio, and the HML premium is the compensation for a risk distinct from market risk that is captured by the performance of the HML portfolio. 
4. As with the key CAPM equation, there is no "alpha" ($\alpha_{i}$) in the key equation describing expected returns under the model. **IF** the model is true, then expected returns vary across stocks/securities *only* due to variation in MKT, SMB, and HML betas. 
5. If the multifactor model is true, then the "tangency" portfolio from mean-variance analysis is no longer the value-weighted market portfolio. If the factors in the model are "tradeable" (as with the Fama-French three-factor model), then the tangency portfolio can be written as a portfolio with positions in the model's tradeable factors, e.g., with positions in the market, HML, and SMB portfolios for the FF 3-factor model. 

*Exactly what "non-market" risks are captured by the SMB and HML factors?* 

This is a good question -- one that has been hotly debated in academic literature ever since Fama and French proposed these factors in the early 1990s. Fama and French suggest that the factors might relate to "distress risk": small firms and value firms might be more susceptible to falling into financial distress. Because there is a common component to this distress risk (it relates to the business cycle), the fact that multiple small or value firms are held in a portfolio does not diversify away the risk. Other researchers suggest alternative explanations. For example, it has been suggested that the SMB premium reflects compensation for *liquidity* (small cap stocks are generally less liquid than large cap stocks). And others believe that the size and value premia may not represent *any* rational risk compensation, but rather reflect persistent *mispricing* related to behavioral biases. 

Similar comments pertain to additional proposed factors included in multifactor models.



## Testing multifactor models

Suppose we want to test key restrictions associated with a multifactor model such as the Fama-French three factor model. One popular way to do so extends the time series regression test that we used for the CAPM. Specifically, we embed the multifactor model in a more general model that permits a potential intercept or $\alpha$ and then we can test the null hypothesis that $\alpha = 0$ for one or more securities or portfolios. 

The regression model is:

$R_{i,t} - R_{f} = \alpha_{i} + \beta_{MKT,i} MKT_{t} + \beta_{SMB,i} SMB_{t} + \beta_{HML,i} HML_{t} + \epsilon_{t}$

This is a *multivariate regression* because there are three right-hand side variables that are purported to explain variation in excess returns for the stock or portfolio of interest on the left-hand side. These variables are the market excess return $MKT_{t}$, the size factor return $SMB_{t}$ and the value factor return $HML_{t}$. 


Because analyzing multifactor models boils down to multivariate regression analysis, it is time to review this statistical topic.



## Multivariate regression

The multivariate regression model can be written in several ways. Perhaps the most intuitive way is:

$Y_{t} = \beta_{0} + \beta_{1} X_{1,t} + \beta_{2} X_{2,t} + ... + \beta_{K} X_{K,t} + \epsilon_{t}$

In the equation above $y_{t}$ is the dependent variable outcome at time $t$, $X_{1,t},...,X_{K,t}$ are the outcomes of $K$ "independent" variables (not necessarily statistically) or predictors, $\beta_{0}$ is an intercept, $\beta_{1},...,\beta_{K}$ are slopes for each of the predictors, and $\epsilon_{t}$ is a regression error term.

### Understanding the model
Similar to the case with a single regressor, we can think of the multivariate linear regression model as model for predicting $y_{t}$, in this case conditional on observing $K$ *covariates*: $X_{1,t}, X_{2,t},...,X_{K,t}$. Once again, in general $Y_{t}$ could be a complicated function of these various covariates, including nonlinear effects, interaction effects, and so forth. The multivariate linear regression model imposes the restriction of predicting $y_{t}$ using a *linear combination* of the covariates. 

The *population prediction function* for the model is $E(Y_{t} | X_{1,t},...,X_{K,t}) = \beta_{0} + \beta_{1}X_{1,t} + ... + \beta_{K}X_{K,t}$.

The interpretation of the parameters $\beta_{1}, \beta_{2},$ etc. is slightly more nuanced. For example, $\beta_{1}$ captures the impact on our prediction of $Y_{t}$ of a one unit change in $X_{1,t}$ *holding constant all of the other covariates* $X_{2,t},...,X_{K,t}$.


In order to use the model, we must estimate the unknown parameters $\beta_{0},...,\beta_{K}$. For this purpose, we will assume that we have a sample $(Y_{t},X_{1,t},...,X_{K,t})$ for $t=1,...,T$, so that $T$ is the sample size. (Sometimes $N$ is used instead, but often in finance the units of observation are over time so I will use $T$). 

It is convenient to re-write the regression equation using matrix notation. Let $Y$ denote the $T \times 1$ vector of dependent variable outcomes, i.e., $Y = (Y_{1},...,Y_{T})^{\prime}$. 

Next, let $X$ equal a $T \times (K+1)$ matrix formed in the following way: the first column is a $T \times 1$ vector of ones, the second column is the $T \times 1$ vector $(X_{1,1},...,X_{1,T})^{\prime}$, the third column is the $T \times 1$ vector $(X_{2,1},...,X_{2,T})^{\prime}$, and so forth with the last column equal to the $T \times 1$ vector $(X_{K,1},...,X_{K,T})^{\prime}$.

The $(K+1) \times 1$ vector $\beta = (\beta_{0},\beta_{1},...,\beta_{K})^{\prime}$. Finally, form the $T \times 1$ vector $\epsilon = (\epsilon_{1},...,\epsilon_{T})^{\prime} $.

With this notation in place, we can now write the regression model more simply as:

$Y = X \beta + \epsilon$

Now, we want to estimate the parameters $\beta$ and we will do so by OLS. Note that the sum of squared residuals, which is the criterion we want to miminize, can be written:

$SSR = \sum_{t=1}^{T} (Y_{t} - X_{t}^{\prime} \beta)^{2} = \sum_{t=1}^{T} \epsilon_{t}^{2} = \epsilon^{\prime}\epsilon = (Y - X\beta)^{\prime} (Y - X \beta)$

The OLS estimator chooses the $\hat{\beta}$ that minimizes the above criterion. Assuming the $X$ has full column rank (this rules out *perfect multicollinearity* meaning that one column is an exact linear combination of other columns), then we can write the OLS estimate that minimizes the sum of squared errors as a relatively simple expression in matrix notation:

$\hat{\beta} = (X^{\prime}X)^{-1} X^{\prime} Y$

The *regression residuals* are then $\hat{\epsilon} = Y - \hat{Y} = MY,$ where $M = I -  X(X^{\prime}X)^{-1} X^{\prime}$

Now, obviously, $Y = \hat{Y} + \hat{\epsilon}$, so that OLS regression decomposes $Y$ into the predicted part (projection onto columns of $X$) and a residual. Less obviously, this decomposition is *orthogonal*:

$\hat{Y}^{\prime} \hat{\epsilon} =  0$ 

This result allows us to make the following important decomposition:

$TSS = ESS + RSS$, where:

$TSS = Y^{\prime} Y$   *total sum of squares*

$ESS = \hat{Y}^{\prime}\hat{Y}$   *explained sum of squares*

$RSS = \hat{\epsilon}^{\prime} \hat{\epsilon}$   *residual sum of squares*

The *uncentered* $R^{2}$-statistic is defined as:

$R^{2}_{UC} = \frac{ESS}{TSS}$

It is common in practice to instead report the *centered* $R^{2}$ value, which is:

$R^{2} = \frac{ESSM}{TSSM}$, where $ESSM = \sum_{1}^{T}(\hat{y}_{t} - \bar{y})^{2}$ is the *centered* explained sum of squares, and $TSSM = \sum_{1}^{T}(y_{t} - \bar{y})^{2}$ is the variation of $y_{t}$ about its mean. Note that $ESSM$ uses deviations of the fitted values from $\bar y$, and is therefore *not* the same quantity as the (uncentered) $ESS$ used above for $R^2_{UC}$.



## Statistical properties of OLS in multivariate setting

So far, everything we have developed is **just linear/matrix algebra**. We have *not* done any "statistics" yet, because we have not made any assumptions about the probability distributions governing $Y$ and $X$ which then determine the statistical properties of the OLS estimates $\hat{\beta}$. 

For the multivariate case, I will not present extensive formal statistical results. Fortunately, these can be viewed as a reasonably straightforward extension of the results we discussed in the univariate regression case. 

In particular, we can make the same set of assumptions in the multivariate case that we did for the univariate case. The only difference is that the $X_{t}$ involved is a $K \times 1$ vector rather than a scalar as in the univariate case. 

The critical assumption for the OLS estimator to have desirable properties is again:
$E(x_{t} \epsilon_{t}) = 0$ 

In this case, the right-hand side is a vector of zeros ($K+1 \times 1$, assuming there is a constant in the model). 

As with the univariate regression case, assuming that $(y_{t}, x_{t}^{\prime})$ is i.i.d. and that $E(\epsilon_{t} | x_{t}) = 0$ ensures that $E(x_{t} \epsilon_{t}) = 0$ holds. Under these assumptions (plus a couple of technical conditions), we have:

$E(\hat{\beta} | X) = \beta$, and since this holds for all potential $X$, $E(\hat{\beta}) = \beta$,  and the OLS estimate is **unbiased**.

The following large-sample or "asymptotic" properties also hold:

$\hat{\beta}$ is *consistent* for $\beta$: in words, the OLS estimate approaches the true parameter value as the sample size tends to infinity

The limiting distribution of $\hat{\beta}$ is (multivariate) normal. We have:

$\sqrt{T}(\hat{\beta} - \beta) \rightarrow \text{MV Normal}(0,V)$

The way to think about the mathematical statement above is that, in large samples, the OLS estimate of $\beta$ approximately follows a multivariate normal with mean equal to $\beta$ (the true parameter value), and a covariance matrix of $(1/T)V$. It is important to understand that $V$ here is a $(K+1) \times (K+1)$ covariance matrix, assuming that the model contains a constant plus $K$ nonconstant regressors. 

What is the limiting covariance matrix $V$ ? This depends on the assumptions we are willing to make about the data and specifically the regression error term. 

*If* we are willing to assume that $E(\epsilon^{2} | x_{t} ) = \sigma^{2}$, which does not depend on $x_{t}$ (an assumption given the fancy name of "homoskedasticity"), then $V = \sigma^{2}Q_{XX}^{-1}$, where $Q_{XX} \equiv E(x_{t}x_{t}^{\prime})$ and this is assumed to be invertible. Just like in the univariate regression case, this limiting variance expression involves "population" quantities (namely $\sigma^{2}$ and $Q_{XX}$). Therefore, we substitute (consistent) sample estimates of these unknown quantities, e.g.,:

$\hat{\sigma}^{2} = \frac{1}{T-(K+1)} \hat{\epsilon}^{\prime} \hat{\epsilon}$, which is the RSS with a degrees-of-freedom adjustment applied. Similarly, we can substitute the estimate $\hat{Q}_{XX} = (1/T) X^{\prime} X$ for the unknown $E(x_{t}x_{t}^{\prime})$. 

Fortunately, Python and other statistical software packages will compute estimates of the covariance matrix of the OLS estimates of $\beta$ for you.  The above is generally the basis for the "classical" standard errors that are reported by such packages. 

The diagonal elements of the estimated covariance matrix are estimates of the variances for each element of $\hat{\beta}$. Square roots of these are standard errors for the elements of $\hat{\beta}$. These can be used to construct confidence intervals and/or $t$-statistics for elements of $\hat{\beta}$. Virtually all OLS regression software packages will print out these standard errors as part of a summary of the regression results. We will see this below for Python examples.






In [1]:
import numpy as np
import pandas as pd
import pandas_datareader.data as web
import datetime as dt
import matplotlib.pyplot as plt
from pandas_datareader.famafrench import get_available_datasets
import warnings
warnings.filterwarnings('ignore')

#Note: we will get our returns data from Ken French's website

#We will look at monthly data for the sorted portfolios, from 1965 through the end of 2025
start = dt.datetime(1965, 1, 1)
end = dt.datetime(2025, 12, 31)

frenchsets = get_available_datasets()
#print(frenchsets)

fffactors = web.DataReader('F-F_Research_Data_Factors', 'famafrench',start,end)

print(fffactors['DESCR'])

ffmfactors = fffactors[0]

ffmfactors.tail(5)


F-F Research Data Factors
-------------------------

This file was created using the 202607 CRSP database. The 1-month TBill rate data until 202405 are from Ibbotson Associates. Starting from 202406, the 1-month TBill rate is from ICE BofA US 1-Month Treasury Bill Index. The annual TBill return is compounded from the monthly T-bill rates from January to December. Copyright 2026 Eugene F. Fama and Kenneth R. French

  0 : (732 rows x 4 cols)
  1 : Annual Factors: January-December (61 rows x 4 cols)


,Mkt-RF,SMB,HML,RF
Date,,,,
2025-08,1.84,3.80,4.37,0.38
2025-09,3.39,-1.83,-1.05,0.33
2025-10,1.96,-0.57,-3.18,0.37
2025-11,-0.13,0.40,3.76,0.30
2025-12,-0.36,-1.04,2.40,0.34


In [2]:
#Compute basic descriptive statistics for the three factors


summarydf = ffmfactors.agg(['min','max','mean','median','std','skew','kurt','count'])
summarydf2 = summarydf.transpose()
summarydf2["annual_mean"] = 12*summarydf2["mean"]
summarydf2["annual_std"] = np.sqrt(12)*summarydf2["std"]
summarydf2["annual_sr"] = np.sqrt(12)*(summarydf2["mean"]/summarydf2["std"])
#manually set SR of risk-free to zero (true by definition)
summarydf2.iloc[3,10] = 0

summarydf2.style.format('{0:,.3f}').set_caption(
    f"Summary Statistics for FF Three Factors and Risk-free Rate, {start.year}-{end.year}."
)



,min,max,mean,median,std,skew,kurt,count,annual_mean,annual_std,annual_sr
Mkt-RF,-23.190,16.110,0.584,0.975,4.507,-0.493,1.658,732.000,7.008,15.613,0.449
SMB,-17.390,21.340,0.157,0.110,3.056,0.415,4.698,732.000,1.887,10.586,0.178
HML,-13.830,12.860,0.278,0.210,2.998,0.106,2.131,732.000,3.332,10.385,0.321
RF,0.000,1.350,0.365,0.380,0.264,0.636,0.708,732.000,4.385,0.916,0.000


In [3]:
#Look at correlations among the factors and risk-free rate

ffmfactors.corr()



,Mkt-RF,SMB,HML,RF
Mkt-RF,1.000000,0.295200,-0.210178,-0.081265
SMB,0.295200,1.000000,-0.143092,-0.040326
HML,-0.210178,-0.143092,1.000000,0.065151
RF,-0.081265,-0.040326,0.065151,1.000000


## Application 1: Nondurables industry portfolio returns

As a first application, let us examine monthly returns on the nondurables industry portfolio from the French Data Library. We will first run a time series regression of nondurables industry portfolio returns on market excess returns (a "single factor" model) and test whether the corresponding intercept (alpha) parameter is zero. *Note*: I am choosing this portfolio because, as we will see, the intercept estimates will differ materially for the single-factor model relative to the three-factor model. 

In [4]:
ff_ind = web.DataReader('10_Industry_Portfolios', 'famafrench',start,end)

ff_indports = ff_ind[0]
ff_indports.head()

#Now we need to convert to excess returns by subtracting the risk-free rate from each raw industry portfolio return
ff_indexret = ff_indports.sub(ffmfactors['RF'],axis="index")
ff_indexret.tail(5)



,NoDur,Durbl,Manuf,Enrgy,HiTec,Telcm,Shops,Hlth,Utils,Other
Date,,,,,,,,,,
2025-08,2.67,7.66,0.98,4.03,0.58,3.42,0.43,4.35,-1.97,3.84
2025-09,-3.39,24.78,2.39,-1.08,6.75,-1.46,-1.48,1.70,3.65,-0.05
2025-10,-3.72,2.19,2.39,-1.18,5.23,-6.22,1.37,3.91,0.07,-3.16
2025-11,4.96,-4.01,-0.59,2.07,-2.82,-1.75,-0.02,10.56,2.60,1.44
2025-12,-1.20,4.05,1.13,-1.14,-0.97,3.37,-1.66,-1.87,-4.61,1.64


### Single-factor (market) model regression

Below we regress excess returns on the nondurables portfolio on market excess returns. The alpha (intercept) coefficient is positive and translates to around 2.2% on an annualized basis. The estimate is statistically significant at the 5% level, though only just: the $t$-statistic is about 1.97 ($p \approx 0.049$).



In [5]:
import statsmodels.api as sm

exmkt = ffmfactors["Mkt-RF"]
yret = ff_indexret["NoDur"] 
regx = sm.add_constant(exmkt)
#Now we reun the regression and look at the results
regresults = sm.OLS(yret,regx).fit()

regresults.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                  NoDur   R-squared:                       0.658
Model:                            OLS   Adj. R-squared:                  0.658
Method:                 Least Squares   F-statistic:                     1406.
Date:                Mon, 14 Sep 2026   Prob (F-statistic):          2.16e-172
Time:                        10:58:05   Log-Likelihood:                -1704.0
No. Observations:                 732   AIC:                             3412.
Df Residuals:                     730   BIC:                             3421.
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.1829      0.093      1.975      0.049       0.001       0.365
Mkt-RF         0.7649      0.020     37.503      0.000       0.725       0.805
==============================================================================
Omnibus:                       30.539   Durbin-Watson:                   1.803
Prob(Omnibus):                  0.000   Jarque-Bera (JB):               80.417
Skew:                           0.108   Prob(JB):                     3.45e-18
Kurtosis:                       4.609   Cond. No.                         4.58
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

### Three-factor model regression

Now we regress the same nondurables portfolio excess returns on the three factors: MKT, SMB, HML using a multivariate regression. Note that the Python code required to run the regression doesn't really change for a multivariate regression: we just include the additional columns in our "sm.add_constant()" line. 

Comments on the results:

1. The alpha coefficient falls in magnitude relative to the CAPM case (from about 0.18\% to about 0.13\% per month), so that the estimated alpha is economically smaller.
2. The new alpha coefficient is not statistically significant, either at the 5\% level or at the 10\% level ($t \approx 1.42$, $p \approx 0.156$).
3. The nondurables portfolio loads *positively* on the HML factor (nondurables tend to exhibit somewhat value characteristics, $t \approx 5.16$) and negatively on the SMB factor (nondurables tend to be somewhat larger by market cap, $t \approx -2.80$). Both loadings are statistically significant. 



In [6]:
#What about the FF 3 factor model? Can it "price" the utilities portfolio? 
regx = sm.add_constant(ffmfactors[["Mkt-RF","SMB","HML"]])
#Now we reun the regression and look at the results
regresults2 = sm.OLS(yret,regx).fit()

regresults2.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                  NoDur   R-squared:                       0.675
Model:                            OLS   Adj. R-squared:                  0.674
Method:                 Least Squares   F-statistic:                     503.9
Date:                Mon, 14 Sep 2026   Prob (F-statistic):          3.94e-177
Time:                        10:58:05   Log-Likelihood:                -1685.8
No. Observations:                 732   AIC:                             3380.
Df Residuals:                     728   BIC:                             3398.
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.1294      0.091      1.419      0.156      -0.050       0.308
Mkt-RF         0.8044      0.021     37.968      0.000       0.763       0.846
SMB           -0.0865      0.031     -2.802      0.005      -0.147      -0.026
HML            0.1587      0.031      5.161      0.000       0.098       0.219
==============================================================================
Omnibus:                       13.549   Durbin-Watson:                   1.787
Prob(Omnibus):                  0.001   Jarque-Bera (JB):               24.146
Skew:                           0.027   Prob(JB):                     5.71e-06
Kurtosis:                       3.888   Cond. No.                         4.84
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

## Testing the Three-Factor Model using Portfolios Sorted by Investment (Asset Growth)

The SMB and HML factors proposed by Fama and French (1993) were designed to remedy the inability of the CAPM to explain variation in returns across stocks related to the market capitalization and book-market characteristics. 

The three-factor model was very influential and remains an important "second benchmark" relative to the CAPM. Over time, however, evidence accumulated that even the three-factor model does not appear to fully explain cross-sectional variation in expected returns. Much of this evidence came from various "anomaly" studies, which documented that expected returns for portfolios formed based on various past price and accounting characteristics did not appear to be fully explained by *either* the CAPM or the three-factor model.

Below, we will replicate one particular finding in this set of studies, which is that variation in expected returns across portfolio formed with respect to "investment" (defined as growth in assets on the balance sheet) is not fully explained by the three-factor model.

We first pull in the new investment-sorted portfolios from the French library, look at expected returns on them, and then run the key time series regressions to determine whether the three-factor model can "price" these portfolios (i.e., whether the null hypothesis of zero alphas can be rejected).

In [7]:
####Pull in portfolios based on investment (asset growth) from French Data Library
ff_INV = web.DataReader('Portfolios_Formed_on_INV', 'famafrench',start,end)

ff_INVports = ff_INV[0]
ff_INVports.head()

#Subtract RF to get excess returns
ff_INVexret = ff_INVports.sub(ffmfactors['RF'],axis="index")
print(ff_INVexret.tail(5))

ff_INVexret.describe()


         Lo 30  Med 40  Hi 30  Lo 20  Qnt 2  Qnt 3  Qnt 4  Hi 20  Lo 10  \
Date                                                                      
2025-08   2.78    3.58   0.51   2.14   4.14   4.26   3.45  -0.53   0.91   
2025-09   0.94    2.95   4.20   2.21  -0.80   3.35   3.71   4.45   6.73   
2025-10  -1.28    1.25   3.05  -0.75  -0.51   2.55  -2.44   4.17   0.10   
2025-11   1.56    0.94  -1.10   1.42   1.10   1.65   1.12  -1.91  -2.56   
2025-12   0.44   -0.09  -0.64  -0.31   1.59  -0.04  -1.16  -0.61   0.23   

         2-Dec  3-Dec  4-Dec  5-Dec  6-Dec  7-Dec  8-Dec  9-Dec  Hi 10  
Date                                                                    
2025-08   2.88   3.81   4.38   2.38   5.42   0.55   5.53   2.55  -2.27  
2025-09  -0.45  -1.06  -0.61   1.89   4.22   4.66   3.05   5.90   3.59  
2025-10  -1.28  -2.15   0.65   2.40   2.64  -2.89  -2.14   5.74   3.22  
2025-11   3.96   1.79   0.62   0.09   2.56  -1.48   2.90   1.40  -3.98  
2025-12  -0.63   1.68   1.53   1.09 

,Lo 30,Med 40,Hi 30,Lo 20,Qnt 2,Qnt 3,Qnt 4,Hi 20,Lo 10,2-Dec,3-Dec,4-Dec,5-Dec,6-Dec,7-Dec,8-Dec,9-Dec,Hi 10
count,732.000000,732.000000,732.000000,732.000000,732.000000,732.000000,732.000000,732.000000,732.000000,732.000000,732.000000,732.000000,732.000000,732.000000,732.000000,732.000000,732.000000,732.000000
mean,0.737227,0.601052,0.561407,0.782773,0.644331,0.597555,0.613566,0.560683,0.786093,0.786311,0.677568,0.621066,0.608197,0.594003,0.631762,0.600738,0.651680,0.455724
std,4.496789,4.158610,5.302208,4.880350,4.136114,4.252263,4.541464,5.687601,5.365359,4.869312,4.364615,4.184583,4.315298,4.388169,4.459357,4.867482,5.440603,6.213968
min,-23.010000,-21.210000,-25.480000,-24.120000,-20.240000,-20.530000,-24.160000,-26.380000,-28.160000,-22.370000,-21.680000,-18.770000,-19.190000,-21.860000,-24.500000,-23.760000,-24.920000,-28.890000
25%,-2.022500,-1.850000,-2.422500,-2.140000,-1.790000,-1.820000,-1.925000,-2.637500,-2.335000,-1.957500,-2.050000,-1.830000,-1.822500,-1.795000,-2.002500,-2.242500,-2.452500,-3.220000
50%,1.060000,0.955000,0.895000,1.085000,0.920000,0.870000,0.880000,1.075000,1.000000,0.965000,0.915000,0.850000,0.855000,0.860000,0.835000,0.645000,1.150000,0.785000
75%,3.455000,3.267500,3.942500,3.790000,3.340000,3.270000,3.502500,4.210000,4.067500,3.795000,3.442500,3.292500,3.087500,3.257500,3.407500,3.655000,4.030000,4.437500
max,16.270000,14.730000,20.430000,16.370000,16.400000,14.380000,17.980000,19.170000,20.060000,18.770000,16.160000,16.590000,14.150000,15.340000,14.520000,22.530000,18.990000,19.460000


### Saving a local backup of the data

Ken French's data library is occasionally slow, reorganized, or briefly unavailable, and the underlying historical figures can also get revised. To make sure this notebook can still be re-run (or the results reproduced) even if the live pull fails, we save local `.csv` backups of the three datasets used above: the factor/risk-free returns, the 10 industry portfolios, and the investment-sorted portfolios.

In [8]:
#Save local CSV backups of the raw data pulled from Ken French's data library
import os

backup_dir = "../data/ff_data_backup"
os.makedirs(backup_dir, exist_ok=True)

pull_date = dt.datetime.today().strftime('%Y-%m-%d')

ffmfactors.to_csv(f"{backup_dir}/ff_factors_{start.year}_{end.year}_pulled_{pull_date}.csv")
ff_indports.to_csv(f"{backup_dir}/ff_10_industry_portfolios_{start.year}_{end.year}_pulled_{pull_date}.csv")
ff_INVports.to_csv(f"{backup_dir}/ff_investment_sorted_portfolios_{start.year}_{end.year}_pulled_{pull_date}.csv")

print(f"Backups saved to {backup_dir}/")

Backups saved to ../data/ff_data_backup/


### Three-factor model regression for a single portfolio
First, let's look do a three-factor model regression for just one of the investment-based portfolios:

In [9]:
#Low asset growth portfolio returns as target
yret = ff_INVexret["Lo 20"] 
#Run multivariate regression on Fama-French 3 factor (excess) returns
regx = sm.add_constant(ffmfactors[["Mkt-RF","SMB","HML"]])
regresults_LoINV = sm.OLS(yret,regx).fit()

regresults_LoINV.summary()


<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                  Lo 20   R-squared:                       0.891
Model:                            OLS   Adj. R-squared:                  0.890
Method:                 Least Squares   F-statistic:                     1979.
Date:                Mon, 14 Sep 2026   Prob (F-statistic):               0.00
Time:                        10:58:05   Log-Likelihood:                -1388.2
No. Observations:                 732   AIC:                             2784.
Df Residuals:                     728   BIC:                             2803.
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.0984      0.061      1.622      0.105      -0.021       0.218
Mkt-RF         1.0121      0.014     71.744      0.000       0.984       1.040
SMB            0.1444      0.021      7.026      0.000       0.104       0.185
HML            0.2539      0.020     12.402      0.000       0.214       0.294
==============================================================================
Omnibus:                       65.217   Durbin-Watson:                   1.899
Prob(Omnibus):                  0.000   Jarque-Bera (JB):              277.958
Skew:                           0.275   Prob(JB):                     4.39e-61
Kurtosis:                       5.968   Cond. No.                         4.84
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

### A more systematic analysis

Now we will analyze whether the CAPM and 3-factor models can 'price' the investment-sorted portfolios. We will focus on the quantile portfolios and a "long-short" or "hedge" portfolio that goes long the lowest quintile investment portfolio and short the highest quintile. The basic idea behind such long-short portfolios is that they are bets on *relative* performance and they are (roughly although not perfectly) hedged against market movements. 

We will:

1. Run regressions of excess returns on the portfolios on the market excess return to test the CAPM (see if the alphas are statistically different from zero).
2. Run regressions of excess returns on the portfolios on the market excess return, SMB return, and HML return to test the 3-factor model (see if the alphas are statistically different from zero).

In [10]:
import statsmodels.formula.api as smf

#Drop everything but the quantile portfolios
inv_df = ff_INVexret[['Lo 20', 'Qnt 2', 'Qnt 3', 'Qnt 4', 'Hi 20']]
#merge in HML factors
inv_df2 = pd.merge(inv_df,ffmfactors,on='Date')

#Now we have a dataset with the targets (first 5 columns), and predictors (next 3 columns) (and RF that we don't need)
#BUT, we need to re-name the columns without spaces and dashes for the regression code below to work nicely, so....
inv_df2.columns = ["Lo","Qnt2","Qnt3","Qnt4","Hi","MKT","SMB","HML","RF"]
print(inv_df2.tail())
inv_df2.describe()

           Lo  Qnt2  Qnt3  Qnt4    Hi   MKT   SMB   HML    RF
Date                                                         
2025-08  2.14  4.14  4.26  3.45 -0.53  1.84  3.80  4.37  0.38
2025-09  2.21 -0.80  3.35  3.71  4.45  3.39 -1.83 -1.05  0.33
2025-10 -0.75 -0.51  2.55 -2.44  4.17  1.96 -0.57 -3.18  0.37
2025-11  1.42  1.10  1.65  1.12 -1.91 -0.13  0.40  3.76  0.30
2025-12 -0.31  1.59 -0.04 -1.16 -0.61 -0.36 -1.04  2.40  0.34

,Lo,Qnt2,Qnt3,Qnt4,Hi,MKT,SMB,HML,RF
count,732.000000,732.000000,732.000000,732.000000,732.000000,732.000000,732.000000,732.000000,732.000000
mean,0.782773,0.644331,0.597555,0.613566,0.560683,0.584016,0.157227,0.277691,0.365437
std,4.880350,4.136114,4.252263,4.541464,5.687601,4.506942,3.056056,2.998020,0.264369
min,-24.120000,-20.240000,-20.530000,-24.160000,-26.380000,-23.190000,-17.390000,-13.830000,0.000000
25%,-2.140000,-1.790000,-1.820000,-1.925000,-2.637500,-2.050000,-1.755000,-1.440000,0.140000
50%,1.085000,0.920000,0.870000,0.880000,1.075000,0.975000,0.110000,0.210000,0.380000
75%,3.790000,3.340000,3.270000,3.502500,4.210000,3.480000,2.020000,1.752500,0.500000
max,16.370000,16.400000,14.380000,17.980000,19.170000,16.110000,21.340000,12.860000,1.350000


In [11]:
#First test CAPM

mod = smf.ols('Lo ~ MKT ', data=inv_df2)
res1 = mod.fit()
print(res1.summary())

mod = smf.ols('Qnt2 ~ MKT', data=inv_df2)
res2 = mod.fit()
print(res2.summary())

mod = smf.ols('Qnt3 ~ MKT', data=inv_df2)
res3 = mod.fit()
print(res3.summary())

mod = smf.ols('Qnt4 ~ MKT ', data=inv_df2)
res4 = mod.fit()
print(res4.summary())

mod = smf.ols('Hi ~ MKT', data=inv_df2)
res5 = mod.fit()
print(res5.summary())

#Long short portfolio
inv_df2['LoHi'] = inv_df2['Lo']- inv_df2['Hi']  
mod = smf.ols('LoHi ~ MKT ', data=inv_df2)
res6 = mod.fit()
print(res6.summary())


                            OLS Regression Results                            
Dep. Variable:                     Lo   R-squared:                       0.862
Model:                            OLS   Adj. R-squared:                  0.862
Method:                 Least Squares   F-statistic:                     4572.
Date:                Mon, 14 Sep 2026   Prob (F-statistic):               0.00
Time:                        10:58:06   Log-Likelihood:                -1472.8
No. Observations:                 732   AIC:                             2950.
Df Residuals:                     730   BIC:                             2959.
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      0.1955      0.068      2.895      0.0

                            OLS Regression Results                            
Dep. Variable:                   Qnt4   R-squared:                       0.948
Model:                            OLS   Adj. R-squared:                  0.947
Method:                 Least Squares   F-statistic:                 1.319e+04
Date:                Mon, 14 Sep 2026   Prob (F-statistic):               0.00
Time:                        10:58:06   Log-Likelihood:                -1066.9
No. Observations:                 732   AIC:                             2138.
Df Residuals:                     730   BIC:                             2147.
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      0.0407      0.039      1.050      0.2

In [12]:

from statsmodels.iolib.summary2 import summary_col

dfoutput_singlefactor = summary_col([res1,res2,res3,res4,res5,res6],stars=True)
print(dfoutput_singlefactor)


                   Lo       Qnt2      Qnt3      Qnt4       Hi       LoHi   
---------------------------------------------------------------------------
Intercept      0.1955*** 0.1376*** 0.0673    0.0407    -0.1440** 0.3395*** 
               (0.0675)  (0.0503)  (0.0431)  (0.0388)  (0.0622)  (0.1085)  
MKT            1.0056*** 0.8676*** 0.9079*** 0.9809*** 1.2066*** -0.2010***
               (0.0149)  (0.0111)  (0.0095)  (0.0085)  (0.0137)  (0.0239)  
R-squared      0.8623    0.8938    0.9261    0.9476    0.9141    0.0884    
R-squared Adj. 0.8621    0.8936    0.9260    0.9475    0.9140    0.0872    
Standard errors in parentheses.
* p<.1, ** p<.05, ***p<.01


In [13]:
#Now we test the 3-factor model


mod = smf.ols('Lo ~ MKT + SMB + HML', data=inv_df2)
res1 = mod.fit()
print(res1.summary())

mod = smf.ols('Qnt2 ~ MKT + SMB + HML', data=inv_df2)
res2 = mod.fit()
print(res2.summary())

mod = smf.ols('Qnt3 ~ MKT + SMB + HML', data=inv_df2)
res3 = mod.fit()
print(res3.summary())

mod = smf.ols('Qnt4 ~ MKT + SMB + HML', data=inv_df2)
res4 = mod.fit()
print(res4.summary())

mod = smf.ols('Hi ~ MKT + SMB + HML', data=inv_df2)
res5 = mod.fit()
print(res5.summary())

#Long short portfolio
inv_df2['LoHi'] = inv_df2['Lo']- inv_df2['Hi']  
mod = smf.ols('LoHi ~ MKT + SMB + HML', data=inv_df2)
res6 = mod.fit()
print(res6.summary())



                            OLS Regression Results                            
Dep. Variable:                     Lo   R-squared:                       0.891
Model:                            OLS   Adj. R-squared:                  0.890
Method:                 Least Squares   F-statistic:                     1979.
Date:                Mon, 14 Sep 2026   Prob (F-statistic):               0.00
Time:                        10:58:06   Log-Likelihood:                -1388.2
No. Observations:                 732   AIC:                             2784.
Df Residuals:                     728   BIC:                             2803.
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      0.0984      0.061      1.622      0.1

                            OLS Regression Results                            
Dep. Variable:                   LoHi   R-squared:                       0.412
Model:                            OLS   Adj. R-squared:                  0.409
Method:                 Least Squares   F-statistic:                     169.9
Date:                Mon, 14 Sep 2026   Prob (F-statistic):           1.74e-83
Time:                        10:58:06   Log-Likelihood:                -1659.4
No. Observations:                 732   AIC:                             3327.
Df Residuals:                     728   BIC:                             3345.
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      0.1251      0.088      1.422      0.1

In [14]:
#This part makes a relatively nice formatted table of results using the summary_col() function from statsmodels
dfoutput = summary_col([res1,res2,res3,res4,res5,res6],stars=True)
print(dfoutput)


                   Lo       Qnt2       Qnt3       Qnt4        Hi        LoHi   
-------------------------------------------------------------------------------
Intercept      0.0984    0.0536     0.0070     0.0539     -0.0266    0.1251    
               (0.0607)  (0.0418)   (0.0367)   (0.0381)   (0.0480)   (0.0879)  
MKT            1.0121*** 0.9169***  0.9528***  0.9933***  1.1366***  -0.1245***
               (0.0141)  (0.0097)   (0.0085)   (0.0088)   (0.0112)   (0.0204)  
SMB            0.1444*** -0.0766*** -0.0994*** -0.0812*** 0.1125***  0.0319    
               (0.0206)  (0.0142)   (0.0124)   (0.0129)   (0.0162)   (0.0298)  
HML            0.2539*** 0.2426***  0.1789***  -0.0275**  -0.3391*** 0.5930*** 
               (0.0205)  (0.0141)   (0.0124)   (0.0128)   (0.0162)   (0.0297)  
R-squared      0.8908    0.9278     0.9474     0.9504     0.9497     0.4118    
R-squared Adj. 0.8903    0.9276     0.9472     0.9502     0.9495     0.4094    
Standard errors in parentheses.
* p<.1,

### Interpreting the results

Let's unpack the results in the tables above. Key findings are:

1. All portfolios load strongly on the MKT factor *except* the 'hedge' portfolio in the last column. Intuitively, the hedge portfolio is roughly market neutral because it is long some stocks (low investment firms) but short other stocks (high investment firms) which dramatically reduces sensitivity to market movements. It should be noted that the market betas are still statistically significant so these long-short portfolios are not perfectly hedged.
2. Using the CAPM, several of the intercept ($\alpha$) estimates are statistically different from zero: the low-investment portfolio ($p \approx 0.004$), the second quintile ($p \approx 0.006$), and the high-investment portfolio ($p \approx 0.021$, with a *negative* alpha) are all significant at the 5\% level, and the alpha on the long-short portfolio is positive and highly significant ($t \approx 3.13$, $p \approx 0.002$). The interpretation is that the CAPM is unable to 'price' these portfolios (it cannot fully explain the variation in average returns across the portfolios).
3. Using the three-factor model, things look considerably better. None of the six individual portfolios has an alpha that is statistically significant at conventional levels — the low-investment portfolio comes closest ($t \approx 1.62$, $p \approx 0.105$), but even that falls just short of the 10\% threshold. The long-short portfolio alpha estimate also shrinks substantially relative to the CAPM case (from about 0.34\% to about 0.13\% per month) and is no longer close to statistically significant ($t \approx 1.42$, $p \approx 0.155$).

*Final note*: This example is a nice illustration of the practical value of moving from the CAPM to a multifactor model: the three-factor model soaks up nearly all of the pricing errors that the CAPM leaves on the table for these investment-sorted portfolios, and it does so not just in this updated 1965-2025 sample but across earlier sample periods as well. That doesn't mean the three-factor model is the last word on multifactor asset pricing, however -- as we show next, it does considerably less well at explaining returns to a different, equally famous anomaly: momentum.




## Beyond the Three-Factor Model: The Momentum Anomaly

We just saw that, in this updated sample, the Fama-French three-factor model actually does a reasonably good job of pricing portfolios sorted on investment (asset growth) -- none of the resulting alphas were statistically significant. Does that mean the three-factor model was the final word on multifactor asset pricing models? Not at all.

Recall from our earlier discussion that *momentum* -- the tendency of stocks that have performed well (poorly) over roughly the past year to keep performing well (poorly) over the next few months -- is one of the most robust and puzzling patterns in the cross-section of stock returns. A natural test of the three-factor model is to ask: can it explain the returns earned by a momentum strategy (long recent "winners," short recent "losers")?

Fama and French themselves examined this question and found that the three-factor model does a *very poor* job of explaining momentum returns. In fact, the alpha on a momentum portfolio, after controlling for MKT, SMB, and HML, tends to be *larger* than the portfolio's raw average return, because the model's factor loadings actually work against it. This failure was one of the key motivations for extending multifactor models beyond the original Fama-French three factors: Mark Carhart (1997) added a momentum factor (often labeled "UMD," for "up minus down," or "WML," for "winners minus losers") to create a four-factor model, and momentum remains a standard factor in many academic and practitioner asset pricing models today.

Other well-known extensions include the profitability and investment factors that Fama and French themselves later added in their five-factor model (2015), as well as the "quality minus junk" factor mentioned earlier in this notebook. The larger point is that the three-factor model was an important advance relative to the CAPM, but it was never intended to be -- and was not -- the last word on factor models.

### Three-factor model regression on the momentum factor

Let's test directly whether the three-factor model can explain the momentum factor's own returns. If the model could fully "explain" momentum, we would expect the regression intercept ($\alpha$) to be economically small and statistically indistinguishable from zero -- just as we found (mostly) for the investment-sorted portfolios above.

In [15]:
#Pull in the momentum ("winners minus losers") factor from the French Data Library
ff_mom = web.DataReader('F-F_Momentum_Factor', 'famafrench', start, end)

mom_df = ff_mom[0]
mom_df.columns = [c.strip() for c in mom_df.columns]  #raw column header has extra whitespace padding
print(mom_df.tail())

#Back up the momentum data too, alongside our other CSV backups
mom_df.to_csv(f"{backup_dir}/ff_momentum_factor_{start.year}_{end.year}_pulled_{pull_date}.csv")

#Merge with the three Fama-French factors (and RF) so everything lines up on the same dates
mom_df2 = pd.merge(mom_df, ffmfactors, on='Date')
mom_df2.columns = ["MOM","MKT","SMB","HML","RF"]
mom_df2.tail()

          Mom
Date         
2025-08 -3.61
2025-09  4.62
2025-10  0.16
2025-11 -1.72
2025-12 -2.40


,MOM,MKT,SMB,HML,RF
Date,,,,,
2025-08,-3.61,1.84,3.80,4.37,0.38
2025-09,4.62,3.39,-1.83,-1.05,0.33
2025-10,0.16,1.96,-0.57,-3.18,0.37
2025-11,-1.72,-0.13,0.40,3.76,0.30
2025-12,-2.40,-0.36,-1.04,2.40,0.34


In [16]:
#Regress the momentum factor itself on MKT, SMB, and HML
#If the three-factor model could "explain" momentum, we'd expect a small, insignificant alpha here
mod = smf.ols('MOM ~ MKT + SMB + HML', data=mom_df2)
res_mom = mod.fit()
print(res_mom.summary())

                            OLS Regression Results                            
Dep. Variable:                    MOM   R-squared:                       0.087
Model:                            OLS   Adj. R-squared:                  0.083
Method:                 Least Squares   F-statistic:                     23.14
Date:                Mon, 14 Sep 2026   Prob (F-statistic):           2.59e-14
Time:                        10:58:07   Log-Likelihood:                -2059.7
No. Observations:                 732   AIC:                             4127.
Df Residuals:                     728   BIC:                             4146.
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      0.8164      0.152      5.373      0.0

### Interpreting the momentum result

The results are striking, and they confirm exactly the pattern the literature has documented. The three-factor model's intercept for the momentum factor is $\hat\alpha \approx 0.82\%$ per month (roughly **9.8% per year**), with $t \approx 5.37$ ($p < 0.001$) -- by far the most statistically significant alpha we have encountered anywhere in this notebook. Rather than shrinking toward zero once we control for MKT, SMB, and HML, the momentum factor's alpha is *larger* than many of its raw average monthly returns would suggest, precisely because its loadings on the three factors point in the "wrong" direction: the momentum factor loads *negatively* on both MKT ($\hat\beta \approx -0.20$, $t \approx -5.77$) and HML ($\hat\beta \approx -0.35$, $t \approx -6.75$), so subtracting off these factor exposures actually *increases* the unexplained intercept rather than shrinking it. The loading on SMB is small and statistically insignificant ($t \approx -0.65$). Notice also that the model's $R^2$ is quite low (under 9\%): the vast majority of the momentum factor's variation is simply not captured by MKT, SMB, and HML.

This is exactly why the three-factor model cannot be the final word on multifactor asset pricing: it leaves a large, robust, and easily-exploited source of average returns on the table. As discussed above, this is precisely the gap that Carhart's four-factor model (adding momentum directly as a fourth factor) was designed to close.